# V8: thử loss feature nhiều tầng, giữ VideoSwinLite

Bật GPU + Internet. Inputs: Output V7 đầy đủ (v7_rate_recovery_20260907_163149_952110 với recovery_split.json, selection.json và proxy_calibrated/best.pt); Output V6 có best_task_bd_rate.pt −6,687%; checking có manifest; kineticscleaned. Có thể dùng ngay file còn trong Working.

Run All: lấy code, khôi phục đúng split V7, đo baseline V6 trên controller V7, thử 2 epoch rồi in so sánh. Không train proxy lại. Công thức mới là thí nghiệm từ các bài trong News, chưa có bằng chứng BD-rate tốt hơn. Giữ target 0.95/lr 1e-5 và mọi thành phần khác của V7; tùy chọn đối chứng cosine ở Cell 3.


In [ ]:
from pathlib import Path
import subprocess, sys, importlib
PROJECT = Path("/kaggle/working/proxy_v3")
if PROJECT.exists():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/munnn01/proxy_v3.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT))
from kaggle_cells import v7_rate_recovery, v8_feature_ablation
importlib.reload(v7_rate_recovery)
v8 = importlib.reload(v8_feature_ablation)
RUN = v8.prepare()


In [ ]:
BASELINE_JSON = v8.baseline(RUN)


In [ ]:
CANDIDATE_PT, IS_FEASIBLE = v8.train(RUN)
print("Candidate:", CANDIDATE_PT, "Feasible:", IS_FEASIBLE)
# Optional matched control, in a separate folder:
# LEGACY_PT, LEGACY_FEASIBLE = v8.train(RUN, mode="legacy")


In [ ]:
import json, torch
from IPython.display import display, FileLink
initial = json.loads(BASELINE_JSON.read_text())["val_metrics"]
candidate = torch.load(CANDIDATE_PT, map_location="cpu", weights_only=False)["val_metrics"]
print("Same controller: V6 initial BD-rate =", initial["task_bd_rate_percent"])
print("Same controller: candidate BD-rate =", candidate["task_bd_rate_percent"])
print("QP | initial BPP ratio | candidate BPP ratio | initial Top1 | candidate Top1")
for qp in [30, 35, 40, 45]:
    print(qp, initial[f"qp{qp}_bpp_ratio"], candidate.get(f"qp{qp}_bpp_ratio"),
          initial[f"qp{qp}_top1"], candidate[f"qp{qp}_top1"])
display(FileLink(str(BASELINE_JSON)))
# Optional full validation at seven QPs, after inspecting the controller result:
# EVAL_DIR = v7_rate_recovery.evaluate_candidate(RUN, CANDIDATE_PT)
